## Building a RAG System with LangChain and ChromaDB
## Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

c:\Users\CHITTA\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\CHITTA\AppData\Local\Temp\ipykernel_19780\2872574895.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model = "gemini-embedding-2-preview"
)

In [6]:
from langchain_community.vectorstores import Chroma

In [7]:
import numpy as np
from typing import List

## RAG (Retrieval-Augmented Generation) Architecture:
1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

### Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge

## 1. Sample Data

In [1]:
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [9]:
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt", "w") as f:
        f.write(doc)
    
print(f"Sample document create in: {temp_dir}")

Sample document create in: C:\Users\CHITTA\AppData\Local\Temp\tmp0y41xlbx


In [8]:
os.makedirs("data", exist_ok = True)

for i, doc in enumerate(sample_docs):
    with open(f"data/doc_{i}.txt", "w") as f:
        f.write(doc)

## 2. Document Loading

In [2]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader(
    "data",
    glob = "*.txt",
    loader_cls = TextLoader,
    loader_kwargs = {'encoding': 'utf-8'}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")

Loaded 3 documents

First document preview:

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther...


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,     # Maximum size of each chunk
    chunk_overlap = 50,   # Overlap between chunks to maintain context
    length_function = len,
    separators = [" ", "\n\n", "\n", "."]    # Hierarchy of separators
)
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experie...
Metadata: {'source': 'data\\doc_0.txt'}


## Embedding Models

In [13]:
sameple_text = "Machine Learing is Facination"
vector = embeddings.embed_query(sameple_text)

### Intilialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [9]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

## Cerate a ChromaDB vector store
persist_directory = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-2-preview"),
    persist_directory = persist_directory,
    collection_name = "rag_collection"
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector store created with 5 vectors
Persisted to: ./chroma_db


## Test Similarity Search

In [ ]:
query = "What are the types of machine learning?"

similar_docs = vectorstore.similarity_search(query, k = 3)
for doc in similar_docs:
    print(doc.page_content)
    print("-" * 50)

In [ ]:
query="what is NLP?"

similar_docs = vectorstore.similarity_search(query, k =  3)
for doc in similar_docs:
    print(doc.page_content)
    print("\n")

In [ ]:
query="what is Deep Learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
for doc in similar_docs:
    print(doc.page_content)
    print("\n")

In [ ]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

## Advanced Similarity Search with Scores

In [ ]:
results = vectorstore.similarity_search_with_score(query, k = 3)
for doc, score in results:
    print(f"Score: {score}")
    print(f"Content: {doc.page_content}")
    print("-" * 50)

### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

## Initialize LLM, RAG Chain, Prompt Template, Query the RAG system

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=2.0,
)


In [ ]:
test_response = llm.invoke("What is machine learning?")
print(test_response.content[0]['text'])

In [ ]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model(
    model="gemini-2.0-flash",
    model_provider="google_genai",
    temperature=0
)

In [20]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="qwen3.5:9b")

In [23]:
test_response = llm.invoke("What is machine learning?")
test_response.content

'At its simplest, **machine learning (ML)** is a branch of artificial intelligence that gives computers the ability to learn from data and experience without being explicitly programmed for every specific task.\n\nInstead of telling a computer exactly how to solve a problem with "if-then" rules, you give it historical data, and the machine identifies patterns within that data to make predictions or decisions.\n\n### The Core Concept: Programming vs. Learning\nTo understand ML, it helps to compare it to traditional programming:\n\n*   **Traditional Programming:** A programmer writes specific instructions (rules).\n    *   *Input:* Raw Data (e.g., a photo) $\\rightarrow$ **Rules** (Programmer) $\\rightarrow$ **Output:** Result (e.g., "This is a cat").\n*   **Machine Learning:** A machine finds the rules itself based on data.\n    *   *Input:* Raw Data (e.g., many photos labeled as "cat") + Output labels $\\rightarrow$ **Algorithm** (ML Model) $\\rightarrow$ **Output:** New prediction (e.

In [ ]:
print(test_response.content)

## Modern RAG Chain

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}      # Number of documents to retrieve
)
retriever

In [27]:
## Create a prompt template
system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

## Create a document chain

##### What is create_stuff_documents_chain?
`create_stuff_documents_chain` creates a chain that "stuffs" (inserts) all retrieved documents into a single prompt and sends it to the `LLM`. It's called "stuff" because it literally stuffs all the documents into the `context` window at once.

In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

## This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

In [ ]:
from langchain_classic.chains import create_retrieval_chain
rag_chain = create_retrieval_chain(retriever, document_chain)
rag_chain

In [30]:
response = rag_chain.invoke({"input": "What is Supervised learning?"})

In [31]:
response['answer']

'Supervised learning is a type of machine learning that uses labeled data to train models. It enables systems to learn and improve from experience without being explicitly programmed for specific tasks. This approach is one of the three main categories alongside unsupervised and reinforcement learning.'

In [ ]:
# Function to query the modern RAG system
def query_rag_modern(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Using create_retrieval_chain approach
    result = rag_chain.invoke({"input": question})
    
    print(f"Answer: {result['answer']}")
    print("\nRetrieved Context:")
    for i, doc in enumerate(result['context']):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")
    
    return result

# Test queries
test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?"
]

for question in test_questions:
    result = query_rag_modern(question)
    print("\n" + "="*80 + "\n")

## Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [49]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [34]:
from langchain_core.prompts import ChatPromptTemplate
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context: {context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [35]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [38]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
## Build the chain ussing LCEL

rag_chain_lcel=(
    { 
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001DF84AB8980>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, model='qwen3.5:9b')
| StrOutputParser()

In [39]:
response = rag_chain_lcel.invoke("What is Deep Learning")
response

'Based on the context provided, Deep learning is defined as **a subset of machine learning based on artificial neural networks**.\n\nSpecific details from the text support this definition further:\n*   **Structure:** The artificial neural networks used in deep learning are inspired by the human brain and consist of layers of interconnected nodes.\n*   **Applications:** Deep learning has revolutionized specific fields such as computer vision, natural language processing, and speech recognition.'

In [44]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")
    
    # Get source documents separately if needed
    docs = retriever.invoke(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [ ]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")

### Add New Documents To Existing Vector Store

In [47]:
from langchain_core.documents import Document

new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

new_doc = Document(
    page_content=new_document,
    metadata={
        "source": "manual_addition", 
        "topic": "reinforcement_learning"
    }
)

In [48]:
new_chunks = text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.')]

In [49]:
### Add new documents to vectorstore
vectorstore.add_documents(new_chunks)

['8a84be96-a27f-42f7-bc28-b2114b3d5cc5',
 '59af4c2d-e196-420f-8e6f-88759dcf19a3']

In [50]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

Added 2 new chunks to the vector store
Total vectors now: 7


In [51]:
new_question = "What are the keys concepts in reinforcement learning"
result = query_rag_lcel(new_question)
result

Question: What are the keys concepts in reinforcement learning
--------------------------------------------------
Answer: Based on the provided context, the key concepts in reinforcement learning are **states, actions, rewards, policies, and value functions**.

This is supported by the following specific detail from the text:
"Key concepts in RL include: states, actions, rewards, policies, and value functions."

Source Documents:

--- Source 1 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or p...

--- Source 2 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 3 ---
methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems....


### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

- `create_history_aware_retriever`: Makes the retriever understand conversation context
- `MessagesPlaceholder`: Placeholder for chat history in prompts
- `HumanMessage`/`AIMessage`: Structured message types for conversation history

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [54]:
from langchain_classic.chains import create_history_aware_retriever

## create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001DF84AB8980>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, T

In [55]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever, 
    question_answer_chain
)
print("Conversational RAG chain created!")

Conversational RAG chain created!


In [57]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history=[]
# First question
result = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
chat_history.extend([
    HumanMessage(content = "What is machine learning?"),
    AIMessage(content = result['answer'])
])
print(f"Q: What is machine learning?")
print(f"A: {result['answer']}")

Q: What is machine learning?
A: Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It encompasses three main types: supervised learning, unsupervised learning, and reinforcement learning. Deep learning is also a significant branch within this field based on artificial neural networks.


In [58]:
## Follow-up question
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})
print(f"\nFollow-up Answer: \n{result2['answer']}")


Follow-up Answer: 
The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning enables agents to learn by interacting with an environment to maximize cumulative rewards over time.


## GROQ API KEY

In [61]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [66]:
from langchain_groq import ChatGroq

llm = ChatGroq(model = "openai/gpt-oss-120b")

In [64]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="groq:openai/gpt-oss-120b",
    model_provider="groq",
    temperature=0.7,
)

In [68]:
result = llm.invoke("What is machine learning?")

In [ ]:
print(result.content)